# Shor's Algorithm and the Quantum Threat to Classical Cryptography

In 1994, mathematician Peter Shor published an algorithm that would reshape the foundations of modern cryptography.
His algorithm demonstrated that a sufficiently large quantum computer could factor integers in polynomial time,
reducing a problem that takes classical computers billions of years to one solvable in hours or minutes.
This is not a theoretical curiosity: the entire security of RSA encryption rests on the assumption that
factoring the product of two large primes is computationally infeasible. Elliptic Curve Cryptography (ECC)
faces a parallel threat through the quantum discrete logarithm problem.

The urgency is compounded by the **Harvest Now, Decrypt Later (HNDL)** threat model. Nation-state
adversaries and sophisticated attackers are already intercepting and storing encrypted traffic today,
betting that future quantum computers will let them decrypt it retroactively. For data with long
retention requirements (healthcare records, government secrets, financial instruments, intellectual property),
this means the threat window is **not** when a cryptographically relevant quantum computer (CRQC) arrives;
it is **today**, the moment the data is transmitted.

The NSA's CNSA 2.0 advisory mandates that all new National Security Systems use post-quantum algorithms
by **2027**, with full transition by 2033. NIST deprecated RSA and ECC for new applications in 2024
and will disallow them entirely after 2035. The migration window is closing.

This notebook walks through Shor's algorithm step by step, visualizes the qubit gap between today's
hardware and what is needed to break RSA, and demonstrates Zipminator's HNDL risk calculator to
quantify the exposure of real-world data assets.

## The Mathematics of Shor's Algorithm

Shor's algorithm converts the factoring problem into a **period-finding** problem, which quantum
computers solve exponentially faster than classical ones. The procedure has four key stages:

**1. Random base selection.** Choose a random integer $a$ where $1 < a < N$ and $\gcd(a, N) = 1$.
If $\gcd(a, N) \neq 1$, we already found a factor and can stop.

**2. Quantum period finding.** The core quantum subroutine finds the period $r$ of the function
$f(x) = a^x \bmod N$. This is the smallest positive integer $r$ such that $a^r \equiv 1 \pmod{N}$.
The quantum circuit prepares a superposition of all $x$ values, computes $a^x \bmod N$ in a second
register via **modular exponentiation**, then applies the **inverse Quantum Fourier Transform (QFT)**
to the first register. Measurement yields a value $y \approx k \cdot 2^n / r$ for some integer $k$.

**3. Classical post-processing.** Use the **continued fractions algorithm** to extract $r$ from the
measured $y/2^n$. The convergents of the continued fraction expansion give candidate periods.

**4. Factor extraction.** If $r$ is even, compute $\gcd(a^{r/2} \pm 1, N)$. With probability
$\geq 1/2$, at least one of these is a non-trivial factor of $N$.

The computational complexity is $O((\log N)^2 (\log \log N)(\log \log \log N))$ for the quantum
part, compared to the best known classical algorithm (General Number Field Sieve) at
$O\left(\exp\left(\left(\frac{64}{9}\right)^{1/3} (\ln N)^{1/3} (\ln \ln N)^{2/3}\right)\right)$.
For RSA-2048, this is the difference between hours on a quantum computer and trillions of years classically.

The quantum advantage comes from **superposition** and **interference**. The QFT maps periodicity
in the computational basis into peaks in the Fourier basis. Measuring these peaks reveals the period
without ever computing $a^x \bmod N$ for each $x$ individually. This is fundamentally impossible
on a classical computer, which must evaluate the function sequentially or in parallel but still
one-value-at-a-time.

## Environment Setup

This notebook uses Qiskit for quantum circuit construction and simulation. If Qiskit is not
installed, the circuit-building cells will not execute, but the analysis and visualization cells
use only `matplotlib` and `numpy` and will run independently.

Install the quantum extras with: `uv pip install zipminator[quantum]`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

QISKIT_AVAILABLE = False
try:
    import qiskit
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import QFT
    from qiskit.visualization import plot_histogram
    QISKIT_AVAILABLE = True
    print(f"Qiskit version: {qiskit.__version__}")
except ImportError:
    print("Qiskit not installed. Circuit cells will be skipped.")
    print("Install with: uv pip install 'qiskit[visualization]'")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.patches import FancyBboxPatch

print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

### Quantum Dark Theme

All figures in this notebook use a consistent dark theme inspired by Zipminator's visual identity.
The palette is designed for maximum readability on dark backgrounds while conveying the quantum
computing context through cyan, violet, and emerald accent colors.

In [ ]:
# ── Quantum Dark Theme ─────────────────────────────────────────────
QDT = {
    'bg':      '#0a0f1e',
    'axes':    '#111827',
    'cyan':    '#22d3ee',
    'violet':  '#a78bfa',
    'emerald': '#34d399',
    'amber':   '#f59e0b',
    'rose':    '#fb7185',
    'text':    '#e2e8f0',
    'muted':   '#94a3b8',
    'grid':    '#1e293b',
}

def apply_quantum_theme(fig, axes):
    """Apply the Zipminator quantum dark theme to a figure."""
    fig.patch.set_facecolor(QDT['bg'])
    if not hasattr(axes, '__iter__'):
        axes = [axes]
    for ax in axes:
        ax.set_facecolor(QDT['axes'])
        ax.tick_params(colors=QDT['text'], which='both')
        ax.xaxis.label.set_color(QDT['text'])
        ax.yaxis.label.set_color(QDT['text'])
        ax.title.set_color(QDT['text'])
        for spine in ax.spines.values():
            spine.set_color(QDT['grid'])
        ax.grid(True, color=QDT['grid'], alpha=0.4, linewidth=0.5)

print("Quantum dark theme loaded.")

## Factoring N = 15 with Shor's Algorithm

N = 15 is the canonical demonstration target for Shor's algorithm for several reasons.
First, it is the smallest composite number that is a product of two distinct odd primes
(3 and 5), making it the simplest non-trivial factoring problem. Second, the modular
exponentiation circuit for $a^x \bmod 15$ has a short period (either 2 or 4 depending on
the base $a$), which means the quantum circuit is compact enough to simulate classically
and run on near-term hardware.

We use base $a = 7$ here. The sequence $7^x \bmod 15$ produces: 1, 7, 4, 13, 1, 7, 4, 13, ...
giving a period $r = 4$. From this: $\gcd(7^{4/2} + 1, 15) = \gcd(50, 15) = 5$ and
$\gcd(7^{4/2} - 1, 15) = \gcd(48, 15) = 3$. The factors are recovered.

The circuit below uses 8 counting qubits (for precision in the QFT) and 4 work qubits
(to hold the modular arithmetic state), totaling 12 qubits. The counting qubits are
initialized in uniform superposition via Hadamard gates, the work register starts as
$|0001\rangle$ (representing 1), and controlled-SWAP operations implement the modular
exponentiation. The inverse QFT extracts the period information into the measurement basis.

In [ ]:
if QISKIT_AVAILABLE:
    # ── Build Shor's circuit for N=15, a=7 ──────────────────────────
    n_count = 8   # counting qubits (precision of QFT)
    n_work  = 4   # work qubits (mod-15 register)
    qc = QuantumCircuit(n_count + n_work, n_count)

    # Step 1: Hadamard on all counting qubits -> uniform superposition
    for q in range(n_count):
        qc.h(q)

    # Step 2: Initialize work register to |1> (identity for multiplication)
    qc.x(n_count)

    # Step 3: Controlled modular exponentiation for a=7 mod 15
    # Each counting qubit q controls a^(2^q) mod 15 applied to the work register.
    # For a=7: 7^1=7, 7^2=4, 7^4=1 mod 15 (period 4)
    # The controlled permutations are implemented via CSWAP gates.
    for q in range(n_count):
        power = 2 ** q
        # 7^(2^q) mod 15 cycles through permutations with period 4
        effective = power % 4
        if effective == 1:   # 7^1 mod 15 = 7 -> specific permutation
            qc.cswap(q, n_count, n_count + 1)
            qc.cswap(q, n_count + 1, n_count + 2)
            qc.cswap(q, n_count + 2, n_count + 3)
        elif effective == 2: # 7^2 mod 15 = 4 -> swap pairs
            qc.cswap(q, n_count, n_count + 2)
            qc.cswap(q, n_count + 1, n_count + 3)
        elif effective == 3: # 7^3 mod 15 = 13 -> reverse permutation
            qc.cswap(q, n_count + 2, n_count + 3)
            qc.cswap(q, n_count + 1, n_count + 2)
            qc.cswap(q, n_count, n_count + 1)
        # effective == 0 -> 7^4 mod 15 = 1 -> identity, no gates needed

    qc.barrier()

    # Step 4: Inverse QFT on counting register
    qc.append(QFT(n_count, inverse=True), range(n_count))

    # Step 5: Measure counting register
    qc.measure(range(n_count), range(n_count))

    print(f"Circuit: {qc.num_qubits} qubits, depth {qc.depth()}, "
          f"{sum(qc.count_ops().values())} gates")
    print(f"Gate breakdown: {dict(qc.count_ops())}")
    print()
    print("Expected measurement outcomes (binary -> phase -> period):")
    print("  00000000 -> 0/256     -> r could be anything")
    print("  01000000 -> 64/256    -> 1/4 -> r = 4  ✓")
    print("  10000000 -> 128/256   -> 1/2 -> r = 2 (sub-period)")
    print("  11000000 -> 192/256   -> 3/4 -> r = 4  ✓")

    # Draw the circuit
    fig = qc.draw('mpl', fold=50, style={'backgroundcolor': QDT['bg']})
    plt.show()
else:
    print("Qiskit not available. Skipping circuit construction.")
    print("The circuit would use 12 qubits (8 counting + 4 work) and ~60 gates.")

## Qubit Requirements: The Chasm Between Toy Examples and Real RSA

Factoring N = 15 on 12 qubits is a compelling demonstration, but it is separated from
breaking real cryptography by an enormous gap. The number of **logical** qubits needed
to run Shor's algorithm on an $n$-bit number scales as roughly $2n + O(\log n)$. For RSA-2048,
that means approximately 4,098 logical qubits.

However, today's quantum hardware is noisy. Each logical qubit must be encoded in many
**physical** qubits using quantum error correction (QEC). Current estimates for the surface
code, the most practical QEC scheme, require roughly 1,000 to 10,000 physical qubits per
logical qubit depending on the target error rate and circuit depth. This puts RSA-2048
at approximately 4 to 20 million physical qubits.

IBM's current largest processor has ~1,100 qubits (Condor, 2023). Their roadmap targets
100,000 qubits by 2033. Even that falls 1-2 orders of magnitude short of what is needed
to break RSA-2048 with current QEC overhead. However, algorithmic improvements (such as
the 2023 result by Regev reducing qubit counts by $\sqrt{n}$, or the 2025 DARPA-funded
research on low-overhead QEC) could close this gap faster than hardware scaling alone.

The figure below visualizes this gap on a logarithmic scale. Green bars indicate targets
achievable with current or near-term hardware. Amber indicates the transition zone.
Red marks what remains firmly out of reach for now, but within plausible 10-15 year horizons.

In [ ]:
# ── Qubit Requirements: Logical vs Physical ──────────────────────────
targets = ['N=15\n(4 bits)', 'N=21\n(5 bits)', 'RSA-1024\n(1024 bits)',
           'RSA-2048\n(2048 bits)', 'RSA-4096\n(4096 bits)']

logical_qubits  = [12, 15, 2_050, 4_098, 8_196]
physical_qubits = [12, 15, 4_000_000, 20_000_000, 100_000_000]

# Color coding: green=achievable, amber=near-future, red=far-future
bar_colors = [QDT['emerald'], QDT['emerald'], QDT['amber'], QDT['rose'], QDT['rose']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
apply_quantum_theme(fig, [ax1, ax2])

# ── Left panel: Logical qubits ──
bars1 = ax1.barh(targets, logical_qubits, color=bar_colors, edgecolor='white',
                 linewidth=0.5, alpha=0.9, height=0.6)
ax1.set_xscale('log')
ax1.set_xlabel('Logical Qubits (log scale)', fontsize=11, fontweight='bold')
ax1.set_title('Logical Qubits Required for Shor\'s Algorithm', fontsize=13,
              fontweight='bold', pad=15)

# Hardware milestone lines
ax1.axvline(x=156, color=QDT['cyan'], linestyle='--', alpha=0.8, linewidth=1.5)
ax1.text(156 * 1.3, 4.3, 'IBM 156q\n(2025)', color=QDT['cyan'], fontsize=8,
         va='center', fontweight='bold')
ax1.axvline(x=1100, color=QDT['violet'], linestyle='--', alpha=0.8, linewidth=1.5)
ax1.text(1100 * 1.3, 4.3, 'IBM Condor\n1,100q (2023)', color=QDT['violet'],
         fontsize=8, va='center', fontweight='bold')

# Value labels on bars
for bar, val in zip(bars1, logical_qubits):
    ax1.text(bar.get_width() * 1.15, bar.get_y() + bar.get_height() / 2,
             f'{val:,}', va='center', color=QDT['text'], fontsize=9, fontweight='bold')

ax1.set_xlim(1, 50_000)

# ── Right panel: Physical qubits ──
bars2 = ax2.barh(targets, physical_qubits, color=bar_colors, edgecolor='white',
                 linewidth=0.5, alpha=0.9, height=0.6)
ax2.set_xscale('log')
ax2.set_xlabel('Physical Qubits (log scale)', fontsize=11, fontweight='bold')
ax2.set_title('Physical Qubits with Error Correction', fontsize=13,
              fontweight='bold', pad=15)

# Hardware milestone lines
ax2.axvline(x=156, color=QDT['cyan'], linestyle='--', alpha=0.8, linewidth=1.5)
ax2.text(156 * 0.15, 0.3, 'IBM 156q (2025)', color=QDT['cyan'], fontsize=8,
         rotation=90, va='bottom', fontweight='bold')
ax2.axvline(x=100_000, color=QDT['amber'], linestyle='--', alpha=0.8, linewidth=1.5)
ax2.text(100_000 * 0.15, 0.3, 'IBM 100K target (2033)', color=QDT['amber'],
         fontsize=8, rotation=90, va='bottom', fontweight='bold')

# Value labels on bars
for bar, val in zip(bars2, physical_qubits):
    label = f'{val:,}' if val < 1_000_000 else f'{val/1_000_000:.0f}M'
    ax2.text(bar.get_width() * 1.15, bar.get_y() + bar.get_height() / 2,
             label, va='center', color=QDT['text'], fontsize=9, fontweight='bold')

ax2.set_xlim(1, 1_000_000_000)

# ── Legend ──
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=QDT['emerald'], label='Achievable today'),
    Patch(facecolor=QDT['amber'], label='Near-future (5-10 yr)'),
    Patch(facecolor=QDT['rose'], label='Far-future (10-20 yr)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=10, frameon=False, labelcolor=QDT['text'],
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('The Quantum Gap: Current Hardware vs. RSA Requirements',
             fontsize=15, fontweight='bold', color=QDT['cyan'], y=1.02)
plt.tight_layout()
plt.show()

## The HNDL Timeline: When Does Encrypted Data Become Vulnerable?

The Harvest Now, Decrypt Later threat does not begin when a cryptographically relevant
quantum computer (CRQC) is built. It begins the moment sensitive data is transmitted over
a channel that an adversary can intercept. The risk equation is straightforward:

$$\text{At risk if: } T_{\text{harvest}} + T_{\text{retention}} > T_{\text{CRQC}}$$

where $T_{\text{harvest}}$ is the year data is captured, $T_{\text{retention}}$ is how long
it must remain secret, and $T_{\text{CRQC}}$ is when a quantum computer capable of breaking
the encryption becomes available.

The timeline below maps the critical milestones. Note how the CNSA 2.0 deadline (2027)
and NIST deprecation (2030) both precede even the most conservative CRQC estimates,
reflecting the position that the migration must happen well before the threat materializes.
Organizations that wait for a working CRQC before migrating will find their historical
data already compromised.

In [ ]:
# ── HNDL Timeline Visualization ─────────────────────────────────────
milestones = [
    (2024, 'NIST PQC Standards\nFinalized (FIPS 203/204/205)', QDT['emerald'], 1.0),
    (2025, 'Current QC Hardware\n~1,100 qubits (IBM Condor)', QDT['cyan'], -1.0),
    (2027, 'CNSA 2.0 Deadline\nAll new NSS must use PQC', QDT['amber'], 1.0),
    (2030, 'NIST Deprecates\nRSA & ECC', QDT['amber'], -1.0),
    (2033, 'IBM 100K Qubit\nRoadmap Target', QDT['violet'], 1.0),
    (2035, 'NIST Disallows\nRSA & ECC Entirely', QDT['rose'], -1.0),
]

fig, ax = plt.subplots(figsize=(16, 5))
apply_quantum_theme(fig, ax)
ax.grid(False)

# Draw the main timeline axis
ax.axhline(y=0, color=QDT['muted'], linewidth=2, alpha=0.6, zorder=1)

# Plot each milestone
for year, label, color, direction in milestones:
    # Vertical connector
    ax.plot([year, year], [0, direction * 0.4], color=color, linewidth=2,
            solid_capstyle='round', zorder=2)
    # Marker on timeline
    ax.scatter(year, 0, s=120, color=color, zorder=3, edgecolors='white', linewidth=1)
    # Label
    va = 'bottom' if direction > 0 else 'top'
    y_text = direction * 0.50
    ax.text(year, y_text, label, ha='center', va=va, fontsize=9,
            color=color, fontweight='bold', linespacing=1.4)
    # Year label
    y_year = direction * 0.42
    ax.text(year, y_year, str(year), ha='center', va=va, fontsize=11,
            color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8, edgecolor='none'))

# HNDL danger zone shading
ax.axvspan(2024, 2035, alpha=0.08, color=QDT['rose'], zorder=0)
ax.text(2029.5, -0.85, 'HNDL DANGER ZONE', ha='center', fontsize=12,
        color=QDT['rose'], fontweight='bold', alpha=0.7,
        style='italic')
ax.text(2029.5, -0.95, 'Data harvested during this period may be decrypted retroactively',
        ha='center', fontsize=8, color=QDT['muted'], style='italic')

# Current year marker
ax.axvline(x=2026, color=QDT['cyan'], linestyle=':', alpha=0.4, linewidth=1)
ax.text(2026, 0.92, 'NOW', ha='center', fontsize=10, color=QDT['cyan'],
        fontweight='bold')

ax.set_xlim(2023, 2036.5)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel('')
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

fig.suptitle('The Post-Quantum Migration Timeline',
             fontsize=15, fontweight='bold', color=QDT['cyan'], y=1.0)
plt.tight_layout()
plt.show()

## HNDL Risk Calculator

Zipminator includes an HNDL risk scoring engine that quantifies exposure based on four factors:

- **Data sensitivity**: Ranges from `public` (weight 0.1) through `confidential` (0.7) to
  `top_secret` (1.0). Higher sensitivity means greater consequences if decrypted.
- **Retention period**: How many years the data must remain confidential. Healthcare records
  (HIPAA) require 50+ years. Financial records (SOX) require 7-30 years. Government secrets
  may require 75+ years or indefinite protection.
- **Current encryption**: Classical algorithms (RSA, AES, ECC) score poorly because they will
  be breakable once a CRQC exists. PQC algorithms (Kyber768, hybrid modes) score highly because
  they resist both classical and quantum attacks.
- **Industry multiplier**: Regulated industries (government 1.5x, healthcare 1.3x, finance 1.4x)
  face higher risk due to compliance requirements and adversary targeting.

The calculator uses NIST/expert consensus estimates placing a CRQC between 2030 (optimistic)
and 2040 (conservative), with 2035 as the moderate baseline. The risk score ranges from 0 to 100,
with levels: LOW (< 25), MEDIUM (25-50), HIGH (50-75), CRITICAL (> 75).

Below we compare three real-world scenarios, each evaluated with both classical and PQC encryption.

In [ ]:
from zipminator.hndl_risk import HNDLCalculator

calc = HNDLCalculator()

# ── Define scenarios ────────────────────────────────────────────────
scenarios = [
    {'name': 'Healthcare (HIPAA)',   'sensitivity': 'top_secret',
     'years': 50, 'industry': 'healthcare'},
    {'name': 'Financial (SOX)',      'sensitivity': 'confidential',
     'years': 30, 'industry': 'finance'},
    {'name': 'Government (Secret)',  'sensitivity': 'top_secret',
     'years': 75, 'industry': 'government'},
]

print('=' * 78)
print(f'{"Scenario":<25} {"Encryption":<12} {"Risk":>6} {"Level":<10} {"Yrs to Break":>12}')
print('=' * 78)

for scenario in scenarios:
    for enc, enc_label in [('aes256', 'AES-256'), ('kyber768', 'Kyber768')]:
        result = calc.calculate(
            data_sensitivity=scenario['sensitivity'],
            retention_years=scenario['years'],
            current_encryption=enc,
            industry=scenario['industry'],
        )
        symbol = '!!!' if result.risk_level == 'CRITICAL' else \
                 '!! ' if result.risk_level == 'HIGH' else \
                 '!  ' if result.risk_level == 'MEDIUM' else '   '
        print(f'{scenario["name"]:<25} {enc_label:<12} '
              f'{result.overall_risk:>5.1f}% {result.risk_level:<10} '
              f'{result.years_until_quantum_break:>8}yr  {symbol}')
    print('-' * 78)

print()
print('Legend: !!! = CRITICAL, !! = HIGH, ! = MEDIUM')
print()
print('Key insight: Kyber768 (ML-KEM-768, FIPS 203) reduces risk dramatically')
print('because quantum computers cannot break lattice-based cryptography.')

## Key Size Comparison: Classical vs Post-Quantum Algorithms

One common objection to PQC adoption is that post-quantum key sizes and ciphertexts are
larger than their classical counterparts. This is true, but the trade-off is well worth it.
ML-KEM-768 (Kyber) has a public key of 1,184 bytes, compared to 256 bytes for RSA-2048.
That is a 4.6x increase in key size, but it provides security against both classical and
quantum adversaries.

For context, a single HD video frame is ~300KB. The difference between a 256-byte RSA key
and a 1,184-byte Kyber key is negligible in any real-world protocol. Network bandwidth is
not the bottleneck; the bottleneck is time. Every month of delayed migration is another
month of data harvested under vulnerable encryption.

The figure below compares key sizes, ciphertext/signature sizes, and security levels for
the most relevant classical and post-quantum algorithms.

In [ ]:
# ── Key Size Comparison ─────────────────────────────────────────────
algorithms = [
    # (name, pk_bytes, ct_or_sig_bytes, category, quantum_safe)
    ('RSA-2048',     256,    256,    'KEM',  False),
    ('RSA-4096',     512,    512,    'KEM',  False),
    ('ECC P-256',     64,     64,    'KEM',  False),
    ('ML-KEM-768',  1184,   1088,   'KEM',  True),
    ('ML-KEM-1024', 1568,   1568,   'KEM',  True),
    ('ECDSA P-256',   64,     64,   'Sig',  False),
    ('ML-DSA-65',   1952,   3293,   'Sig',  True),
    ('SLH-DSA-128f', 32,   17088,  'Sig',  True),
]

names    = [a[0] for a in algorithms]
pk_sizes = [a[1] for a in algorithms]
ct_sizes = [a[2] for a in algorithms]
q_safe   = [a[4] for a in algorithms]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
apply_quantum_theme(fig, [ax1, ax2])

x = np.arange(len(names))
width = 0.55

# Colors: cyan for quantum-safe, rose for quantum-vulnerable
pk_colors = [QDT['cyan'] if qs else QDT['rose'] for qs in q_safe]
ct_colors = [QDT['violet'] if qs else QDT['amber'] for qs in q_safe]

# Public key sizes
bars1 = ax1.bar(x, pk_sizes, width, color=pk_colors, edgecolor='white',
               linewidth=0.5, alpha=0.9)
ax1.set_ylabel('Bytes', fontsize=11, fontweight='bold')
ax1.set_title('Public Key Size', fontsize=13, fontweight='bold', pad=10)
ax1.set_yscale('log')
for i, (bar, val) in enumerate(zip(bars1, pk_sizes)):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.15,
             f'{val:,}B', ha='center', va='bottom', fontsize=8,
             color=QDT['text'], fontweight='bold')

# Ciphertext / signature sizes
bars2 = ax2.bar(x, ct_sizes, width, color=ct_colors, edgecolor='white',
               linewidth=0.5, alpha=0.9)
ax2.set_ylabel('Bytes', fontsize=11, fontweight='bold')
ax2.set_title('Ciphertext / Signature Size', fontsize=13, fontweight='bold', pad=10)
ax2.set_yscale('log')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=30, ha='right', fontsize=10)
for i, (bar, val) in enumerate(zip(bars2, ct_sizes)):
    label = f'{val:,}B' if val < 10000 else f'{val/1000:.1f}KB'
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.15,
             label, ha='center', va='bottom', fontsize=8,
             color=QDT['text'], fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=QDT['cyan'], label='Quantum-safe (PK)'),
    Patch(facecolor=QDT['rose'], label='Quantum-vulnerable (PK)'),
    Patch(facecolor=QDT['violet'], label='Quantum-safe (CT/Sig)'),
    Patch(facecolor=QDT['amber'], label='Quantum-vulnerable (CT/Sig)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=9, frameon=False, labelcolor=QDT['text'],
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Algorithm Comparison: Key & Ciphertext Sizes',
             fontsize=15, fontweight='bold', color=QDT['cyan'], y=1.01)
plt.tight_layout()
plt.show()

## Conclusion: The Time to Act Is Now

This notebook demonstrated that Shor's algorithm is not science fiction. The mathematics
are proven, the circuits are well-understood, and the only barrier is hardware scale.
That barrier is shrinking every year, with billions in public and private investment
accelerating quantum computing development worldwide.

The data you encrypt today with RSA-2048 or AES-256 may well be decryptable within its
retention period. For healthcare records (50+ years), government secrets (75+ years), and
financial instruments (30+ years), the HNDL risk is already CRITICAL when using classical
cryptography alone.

Zipminator implements **ML-KEM-768 (FIPS 203)**, the NIST-standardized post-quantum key
encapsulation mechanism, verified against official KAT test vectors. It is not a future
product; it is available today as a Python SDK, a Rust core, and a cross-platform super-app.

The migration window between NIST standardization (2024) and RSA deprecation (2030) is
only six years. For regulated industries, the CNSA 2.0 deadline of 2027 is even tighter.
Every month of delay is another month of data exposed to HNDL attacks.

```
pip install zipminator          # Core PQC encryption
pip install zipminator[quantum] # Quantum extras (Qiskit, entropy harvesting)
```